# 01 – Cloud Engineering Prompting Foundations

## Objective

Understand how to design reliable prompts for LLM-based
cloud engineering request classification.

The project focuses on Google Cloud Platform (GCP),
Kubernetes, Cloud Run, Terraform, and Cloud SQL.

## Concepts

- Role
- Task
- Context
- Constraints
- Examples
- Output Format

## Implementation Plan

1. Build a zero-shot cloud request classifier
2. Extend the classifier with few-shot examples
3. Compare zero-shot and few-shot strategies
4. Develop a reusable cloud prompt builder
5. Implement routing priority rules
6. Evaluate classification accuracy and output reliability

## Architecture Goal

Build the foundation for an AI-assisted Cloud Operations
Request Router that could later be integrated into
an Enterprise AI / MLOps Platform.

In [1]:
from prompt_helpers import client_and_model

client, model, provider, dry_reason = client_and_model()

print("Provider:", provider.name)
print("Model:", model)
print("Mode:", "LIVE" if client is not None else "DRY RUN")

if dry_reason:
    print("Reason:", dry_reason)

Provider: groq
Model: openai/gpt-oss-20b
Mode: LIVE


## Zero-Shot Prompting

Zero-shot prompting means asking an LLM to perform a task
without providing examples.

### Use Case: Cloud Engineering Request Classification

Classify technical requests related to cloud infrastructure,
DevOps, and platform engineering.

### Available Categories

- KUBERNETES
- CLOUD_RUN
- TERRAFORM
- DATABASE
- ARCHITECTURE
- OTHER

### Objective

Evaluate whether an LLM can correctly classify cloud
engineering requests without providing labeled examples.

In [2]:
from prompt_helpers import chat

query = "My GKE pods are stuck in Pending state."

prompt = f"""
You are a senior Cloud and DevOps Engineer specializing
in Google Cloud Platform (GCP).

Classify the following cloud engineering request
into exactly one category:

KUBERNETES
CLOUD_RUN
TERRAFORM
DATABASE
ARCHITECTURE
OTHER

Classification rules:
- KUBERNETES: GKE, pods, deployments, Kubernetes networking.
- CLOUD_RUN: Serverless containers, Cloud Run services.
- TERRAFORM: Infrastructure as Code, Terraform state, modules.
- DATABASE: Cloud SQL, PostgreSQL, database connectivity.
- ARCHITECTURE: Cloud system design, scalability, high availability.
- OTHER: Requests outside these categories.

Cloud engineering request: {query}

Return only the category name.
"""

response = chat(
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0
)

print("Query:", query)
print("Prediction:", response)

Query: My GKE pods are stuck in Pending state.
Prediction: KUBERNETES


In [3]:
test_queries = [
    "My Cloud Run service returns HTTP 503 errors.",
    "Terraform plan fails because the state file is locked.",
    "Cloud SQL PostgreSQL cannot establish a private IP connection.",
    "How should I design a highly available multi-region GCP architecture?",
    "How do I change my GitHub profile picture?"
]

for test_query in test_queries:
    test_prompt = prompt.replace(
        query,
        test_query
    )

    prediction = chat(
        messages=[
            {"role": "user", "content": test_prompt}
        ],
        temperature=0
    )

    print(f"Query: {test_query}")
    print(f"Prediction: {prediction}")
    print("-" * 50)

Query: My Cloud Run service returns HTTP 503 errors.
Prediction: CLOUD_RUN
--------------------------------------------------
Query: Terraform plan fails because the state file is locked.
Prediction: TERRAFORM
--------------------------------------------------
Query: Cloud SQL PostgreSQL cannot establish a private IP connection.
Prediction: DATABASE
--------------------------------------------------
Query: How should I design a highly available multi-region GCP architecture?
Prediction: ARCHITECTURE
--------------------------------------------------
Query: How do I change my GitHub profile picture?
Prediction: OTHER
--------------------------------------------------


## Few-Shot Prompting

Few-shot prompting provides labeled examples to guide
the model's classification behavior.

Goal: Improve consistency and clarify category boundaries.

In [4]:
few_shot_examples = [
    (
        "My GKE pod is stuck in CrashLoopBackOff.",
        "KUBERNETES"
    ),
    (
        "My Cloud Run service fails to start after deployment.",
        "CLOUD_RUN"
    ),
    (
        "Terraform apply fails because the state is locked.",
        "TERRAFORM"
    ),
    (
        "Cloud SQL PostgreSQL connections are timing out.",
        "DATABASE"
    ),
    (
        "How can I design a highly available GCP infrastructure?",
        "ARCHITECTURE"
    ),
]

example_block = "\n\n".join(
    f"Query: {message}\nCategory: {category}"
    for message, category in few_shot_examples
)

query = "My GKE deployment cannot schedule pods due to insufficient CPU."

few_shot_prompt = f"""
You are a senior Cloud and DevOps Engineer specializing in GCP.

Classify the following cloud engineering request into
exactly one category:

KUBERNETES
CLOUD_RUN
TERRAFORM
DATABASE
ARCHITECTURE
OTHER

Examples:
{example_block}

Query: {query}
Category:
"""

response = chat(
    messages=[
        {"role": "user", "content": few_shot_prompt}
    ],
    temperature=0
)

print("Query:", query)
print("Few-shot prediction:", response)

Query: My GKE deployment cannot schedule pods due to insufficient CPU.
Few-shot prediction: Category: KUBERNETES


In [5]:
improved_prompt = few_shot_prompt + """

Important:
Return ONLY the category name.

Do not include:
- "Category:"
- Explanations
- Additional text
"""

improved_response = chat(
    messages=[
        {"role": "user", "content": improved_prompt}
    ],
    temperature=0
)

print("Original few-shot:", response)
print("Improved few-shot:", improved_response)

Original few-shot: Category: KUBERNETES
Improved few-shot: KUBERNETES


## Zero-Shot vs. Few-Shot Comparison

Compare both prompting strategies using identical cloud
engineering requests.

### Evaluation Criteria

- Classification correctness
- Output format compliance
- Consistency

### Objective

Evaluate whether zero-shot and few-shot prompting can
correctly classify technical requests related to:

- Kubernetes / GKE
- Cloud Run
- Terraform
- Cloud SQL / Databases
- Cloud Architecture
- Other requests

In [6]:
def build_cloud_prompt(query: str, examples=None) -> str:
    categories = """
KUBERNETES
CLOUD_RUN
TERRAFORM
DATABASE
ARCHITECTURE
OTHER
"""

    example_section = ""

    if examples:
        example_section = "\n\n".join(
            f"Query: {message}\nCategory: {category}"
            for message, category in examples
        )

    return f"""
You are a senior Cloud and DevOps Engineer specializing in GCP.

Classify the following cloud engineering request
into exactly one category:

{categories}

Classification rules:
- KUBERNETES: GKE, pods, deployments, Kubernetes networking.
- CLOUD_RUN: Cloud Run services and serverless containers.
- TERRAFORM: Infrastructure as Code, state files, modules.
- DATABASE: Cloud SQL, PostgreSQL, database connectivity.
- ARCHITECTURE: System design, scalability, high availability.
- OTHER: Requests outside these categories.

Routing priority:
- Classify by the primary technical issue.
- Database connection failures belong to DATABASE.
- Terraform execution failures belong to TERRAFORM.
- Container runtime and deployment failures belong to their platform.
- Use ARCHITECTURE for design and system-level questions.
- Use OTHER when no technical category applies.

{f"Examples:{chr(10)}{example_section}" if examples else ""}

Cloud engineering request:
{query}

Return ONLY the category name.
Do not include explanations or additional text.
"""


In [7]:
test_cases = [
    (
        "My GKE pods are stuck in ImagePullBackOff.",
        "KUBERNETES"
    ),
    (
        "My Cloud Run service returns HTTP 503 after deployment.",
        "CLOUD_RUN"
    ),
    (
        "Terraform plan shows unexpected infrastructure changes.",
        "TERRAFORM"
    ),
    (
        "Cloud SQL PostgreSQL refuses private IP connections.",
        "DATABASE"
    ),
    (
        "How should I design a disaster recovery strategy across GCP regions?",
        "ARCHITECTURE"
    ),
    (
        "How do I change my GitHub profile picture?",
        "OTHER"
    ),
]

print(f"Loaded {len(test_cases)} cloud engineering test cases.")

Loaded 6 cloud engineering test cases.


In [8]:

for test_query, expected in test_cases:


    zero_result = chat(
        messages=[
            {
                "role": "user",
                "content": build_cloud_prompt(test_query)
            }
        ],
        temperature=0
    ).strip()

    few_result = chat(
        messages=[
            {
                "role": "user",
                "content": build_cloud_prompt(
                    test_query,
                    examples=few_shot_examples
                )
            }
        ],
        temperature=0
    ).strip()

    print(f"Query: {test_query}")
    print(f"Expected:  {expected}")
    print(f"Zero-shot: {zero_result}")
    print(f"Few-shot:  {few_result}")
    print("-" * 55)

Query: My GKE pods are stuck in ImagePullBackOff.
Expected:  KUBERNETES
Zero-shot: KUBERNETES
Few-shot:  KUBERNETES
-------------------------------------------------------
Query: My Cloud Run service returns HTTP 503 after deployment.
Expected:  CLOUD_RUN
Zero-shot: CLOUD_RUN
Few-shot:  CLOUD_RUN
-------------------------------------------------------
Query: Terraform plan shows unexpected infrastructure changes.
Expected:  TERRAFORM
Zero-shot: TERRAFORM
Few-shot:  TERRAFORM
-------------------------------------------------------
Query: Cloud SQL PostgreSQL refuses private IP connections.
Expected:  DATABASE
Zero-shot: DATABASE
Few-shot:  DATABASE
-------------------------------------------------------
Query: How should I design a disaster recovery strategy across GCP regions?
Expected:  ARCHITECTURE
Zero-shot: ARCHITECTURE
Few-shot:  ARCHITECTURE
-------------------------------------------------------
Query: How do I change my GitHub profile picture?
Expected:  OTHER
Zero-shot: OTHER


In [9]:
routing_test_cases = [
    (
        "My Cloud Run service cannot connect to Cloud SQL.",
        "DATABASE"
    ),
    (
        "My GKE pod cannot connect to PostgreSQL.",
        "DATABASE"
    ),
    (
        "Terraform apply fails while provisioning a GKE cluster.",
        "TERRAFORM"
    ),
    (
        "My Cloud Run container fails to start after deployment.",
        "CLOUD_RUN"
    ),
]

for test_query, expected in routing_test_cases:

    prediction = chat(
        messages=[
            {
                "role": "user",
                "content": build_cloud_prompt(test_query)
            }
        ],
        temperature=0
    ).strip()

    print(f"Query: {test_query}")
    print(f"Expected: {expected}")
    print(f"Predicted: {prediction}")
    print(f"Correct: {prediction == expected}")
    print("-" * 55)

Query: My Cloud Run service cannot connect to Cloud SQL.
Expected: DATABASE
Predicted: DATABASE
Correct: True
-------------------------------------------------------
Query: My GKE pod cannot connect to PostgreSQL.
Expected: DATABASE
Predicted: DATABASE
Correct: True
-------------------------------------------------------
Query: Terraform apply fails while provisioning a GKE cluster.
Expected: TERRAFORM
Predicted: TERRAFORM
Correct: True
-------------------------------------------------------
Query: My Cloud Run container fails to start after deployment.
Expected: CLOUD_RUN
Predicted: CLOUD_RUN
Correct: True
-------------------------------------------------------


In [10]:
results = []

for strategy, examples in [
    ("Zero-Shot", None),
    ("Few-Shot", few_shot_examples)
]:
    for test_query, expected in test_cases:

        prediction = chat(
            messages=[
                {
                    "role": "user",
                    "content": build_cloud_prompt(
                        test_query,
                        examples=examples
                    )
                }
            ],
            temperature=0
        ).strip()

        results.append({
            "strategy": strategy,
            "expected": expected,
            "prediction": prediction,
            "correct": prediction == expected,
            "valid": prediction in [
                "KUBERNETES",
                "CLOUD_RUN",
                "TERRAFORM",
                "DATABASE",
                "ARCHITECTURE",
                "OTHER"
            ]
        })

for strategy in ["Zero-Shot", "Few-Shot"]:
    strategy_results = [
        r for r in results
        if r["strategy"] == strategy
    ]

    accuracy = sum(
        r["correct"] for r in strategy_results
    ) / len(strategy_results)

    format_compliance = sum(
        r["valid"] for r in strategy_results
    ) / len(strategy_results)

    print(f"\nStrategy: {strategy}")
    print(f"Accuracy: {accuracy:.1%}")
    print(f"Format Compliance: {format_compliance:.1%}")


Strategy: Zero-Shot
Accuracy: 100.0%
Format Compliance: 100.0%

Strategy: Few-Shot
Accuracy: 100.0%
Format Compliance: 100.0%


## Reusable Prompt Builder

Build reusable prompt templates for zero-shot and
few-shot classification.

Goals:
- Separate prompt construction from LLM execution
- Avoid hardcoded string replacements
- Support multiple prompting strategies

In [11]:
CATEGORIES = [
    "KUBERNETES",
    "CLOUD_RUN",
    "TERRAFORM",
    "DATABASE",
    "ARCHITECTURE",
    "OTHER",
]

# Reuse the existing build_cloud_prompt() function

query = "My Cloud Run service cannot connect to Cloud SQL."

zero_shot = build_cloud_prompt(query)

few_shot = build_cloud_prompt(
    query,
    examples=few_shot_examples
)

print("ZERO-SHOT PROMPT:")
print(zero_shot)

print("\nFEW-SHOT PROMPT:")
print(few_shot)

ZERO-SHOT PROMPT:

You are a senior Cloud and DevOps Engineer specializing in GCP.

Classify the following cloud engineering request
into exactly one category:


KUBERNETES
CLOUD_RUN
TERRAFORM
DATABASE
ARCHITECTURE
OTHER


Classification rules:
- KUBERNETES: GKE, pods, deployments, Kubernetes networking.
- CLOUD_RUN: Cloud Run services and serverless containers.
- TERRAFORM: Infrastructure as Code, state files, modules.
- DATABASE: Cloud SQL, PostgreSQL, database connectivity.
- ARCHITECTURE: System design, scalability, high availability.
- OTHER: Requests outside these categories.

Routing priority:
- Classify by the primary technical issue.
- Database connection failures belong to DATABASE.
- Terraform execution failures belong to TERRAFORM.
- Container runtime and deployment failures belong to their platform.
- Use ARCHITECTURE for design and system-level questions.
- Use OTHER when no technical category applies.



Cloud engineering request:
My Cloud Run service cannot connect to C

In [12]:
query = "My Cloud Run service cannot connect to Cloud SQL."

for strategy, examples in [
    ("Zero-Shot", None),
    ("Few-Shot", few_shot_examples),
]:
    prediction = chat(
        messages=[
            {
                "role": "user",
                "content": build_cloud_prompt(
                    query,
                    examples=examples
                )
            }
        ],
        temperature=0
    ).strip()

    print(f"Strategy: {strategy}")
    print(f"Prediction: {prediction}")
    print(f"Valid category: {prediction in CATEGORIES}")
    print("-" * 50)

Strategy: Zero-Shot
Prediction: DATABASE
Valid category: True
--------------------------------------------------
Strategy: Few-Shot
Prediction: DATABASE
Valid category: True
--------------------------------------------------


## Conclusion & Lessons Learned

### Project Overview

Developed an LLM-based Cloud Engineering Request Classifier
using Groq and the openai/gpt-oss-20b model.

The classifier routes technical requests into six categories:

- KUBERNETES
- CLOUD_RUN
- TERRAFORM
- DATABASE
- ARCHITECTURE
- OTHER

### What We Implemented

1. Zero-shot cloud engineering classification
2. Few-shot classification using labeled GCP examples
3. Reusable prompt builder (`build_cloud_prompt`)
4. Routing priority rules for cross-service issues
5. Live LLM inference with temperature=0
6. Automated accuracy and output format evaluation

### Evaluation Results

| Metric | Zero-Shot | Few-Shot |
|---|---|---|
| Model | openai/gpt-oss-20b | openai/gpt-oss-20b |
| Evaluation cases | 6 | 6 |
| Accuracy | 100% | 100% |
| Format compliance | 100% | 100% |

Additional routing priority tests: 4/4 passed.

### Key Learnings

- Zero-shot prompting performed as well as few-shot prompting
  on our small evaluation dataset.
- Explicit output constraints improved format compliance.
- Routing rules help resolve cross-service classification ambiguity.
- Reusable prompt builders prevent errors caused by string replacement.
- Temperature=0 reduces output variability but does not guarantee
  deterministic behavior.
- Classification identifies a routing category, not the root cause
  of an infrastructure problem.

### Limitations

- Evaluation dataset contains only six basic cases.
- Routing priority tests cover only four scenarios.
- Prompt instructions do not guarantee valid output.
- Latency and API token consumption were not measured.
- Complex and previously unseen infrastructure problems require
  further evaluation.

### Next Steps

Notebook 02 – Structured Outputs

- JSON response formats
- Pydantic schema validation
- Structured cloud incident classification
- Severity and confidence fields
- Error handling and invalid output detection

### Future Architecture

Extend the classifier into an AI-assisted Cloud Operations
Request Router integrated with an Enterprise AI/MLOps Platform.